# 04 — Local-shadow Development Evaluation

Evaluate this run's selected detector on its own policy's development (test) split with native FP32 Ultralytics validation, saving per-image matches for dark/bright and Apply/Skip cohorts.

**Inputs:** runs/v2-YYYY-MM-DD/train/weights/best.pt and the run's prepared data.

**Outputs:** runs/v2-YYYY-MM-DD/eval/{metrics.json,paired_stats.npz,plots} only when ALLOW_INFERENCE=True.

**Prerequisites:** Completed 02 and 03. Evaluation never imports or invokes training.

**Outline**

- [1. Configuration](#section-1)
- [2. Policy–Detector Pair](#section-2)
- [3. Evaluation Summary](#section-3)

<a id="section-1"></a>

## 1. Configuration

RUN_DATE=None uses the newest run. Saved metrics are reused when inputs match.

In [1]:
import os

import pandas as pd
from IPython.display import display

from vehicle_detection.common import project_root, require_paths
from vehicle_detection.evaluation import evaluate_experiment
from vehicle_detection.experiment_data import load_experiment

ROOT = project_root()
os.chdir(ROOT)  # Run All behaves the same from any kernel working directory.
EXPERIMENT = "v2"
require_paths(
    config=ROOT / f"notebooks/experiments/{EXPERIMENT}/{EXPERIMENT}.yaml",
    shared=ROOT / "configs/data.yaml",
)
config = load_experiment(ROOT, EXPERIMENT)

RUN_DATE = None
ALLOW_INFERENCE = True
display(pd.Series(config["evaluation"], name="evaluation"))

imgsz             640
batch              32
device              0
workers             0
conf            0.001
iou               0.7
max_det           300
quantize           32
nms             False
agnostic_nms    False
rect             True
augment         False
plots            True
verbose         False
Name: evaluation, dtype: object

<a id="section-2"></a>

## 2. Policy–Detector Pair

Only development is evaluated; the final holdout is never read.

In [2]:
try:
    result = evaluate_experiment(ROOT, EXPERIMENT, RUN_DATE, allow_inference=ALLOW_INFERENCE)
except FileNotFoundError as error:
    result = None
    print(error)
if result:
    display(pd.DataFrame([result["overall"]]))
    display(pd.DataFrame(result["classes"]).set_index("Class"))

Ultralytics 8.4.198 🚀 Python-3.11.14 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16284MiB)
YOLO26n summary (fused): 122 layers, 2,390,707 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4797.1±1706.2 MB/s, size: 109.5 KB)
val: Scanning /home/truong51972/projects/vehicle-detection/runs/v2-2026-11-09/data/labels/test... 13975 images, 121 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13975/13975 1.3Kit/s 10.6s0.1s
val: New cache created: /home/truong51972/projects/vehicle-detection/runs/v2-2026-11-09/data/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 437/437 4.5it/s 1:370.3sss
                   all      13975     152774      0.579       0.39       0.42      0.255
Speed: 0.2ms preprocess, 1.1ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /home/truong51972/projects/vehicle-detection/runs/v2-2026-11-09/eval


,precision,recall,mAP50,mAP50_95
0,0.578975,0.389667,0.419615,0.254724


,Images,Instances,Box-P,Box-R,Box-F1,mAP50,mAP75,mAP50-95
Class,,,,,,,,
car,13820,142541,0.696709,0.671223,0.683728,0.718352,0.422445,0.427384
truck,3773,5965,0.552198,0.452137,0.497183,0.456566,0.347699,0.317533
bus,1759,2276,0.594754,0.388401,0.469922,0.429259,0.354423,0.316910
motorcycle,435,562,0.510901,0.185053,0.271696,0.208801,0.061380,0.091421
bicycle,872,1430,0.540315,0.251520,0.343254,0.285099,0.086315,0.120375


<a id="section-3"></a>

## 3. Evaluation Summary

Cohort AP keeps every GT of the selected images; dark means raw brightness < 0.12.

In [3]:
if result:
    display(
        pd.DataFrame(
            {
                k: {"images": v["images"], "mAP50_95": v["mAP50_95"]}
                for k, v in result["cohorts"].items()
            }
        ).T
    )
    print("Run:", result["inputs"]["run"], "| speed (ms):", result["speed_ms"])
print("Final holdout stays sealed.")

,images,mAP50_95
dark,3307.0,0.223322
bright,10668.0,0.260236


Run: runs/v2-2026-11-09 | speed (ms): {'preprocess': 0.19513252314904295, 'inference': 1.105828590268316, 'loss': 0.00018514704862226315, 'postprocess': 0.11043729352411426}
Final holdout stays sealed.
